# Purchase Intent Model
Condensed version. Part 1 predicts Purchase_Probability + Purchase_Value (regression), Part 2 predicts the High/Medium/Low category (classification).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               RandomForestClassifier, GradientBoostingClassifier)
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

df = pd.read_csv("../data/processed/cleaned_users.csv")
print(df.shape)


(12000, 25)


## Part 1 - Regression (Purchase_Probability, Purchase_Value)
Only using real "known before they decide" inputs - no Booking_Done / Purchase_Intent,
those are outcomes not inputs. Purchase_Value gets zeroed out where Purchased == No
since some "No" rows have noisy nonzero values in the raw data.

In [ ]:
input_features = ["Age", "Monthly_Income", "Annual_Income", "Family_Size",
                   "Plot_Budget", "Preferred_Plot_Size_SqFt", "Distance_to_City_Center_km",
                   "Gender", "Occupation", "City", "Current_Housing_Status",
                   "Preferred_Location", "Purpose", "Loan_Required",
                   "Expected_Purchase_Timeline", "Lead_Source",
                   "Previous_Enquiry", "Site_Visit", "Negotiation_Done"]
cat_cols = ["Gender", "Occupation", "City", "Current_Housing_Status", "Preferred_Location",
            "Purpose", "Loan_Required", "Expected_Purchase_Timeline", "Lead_Source",
            "Previous_Enquiry", "Site_Visit", "Negotiation_Done"]
target_cols = ["Purchase_Probability", "Purchase_Value"]

Linear Regression: avg R2=0.616  (Probability=0.841, Value=0.391)


Tuned Random Forest: avg R2=0.647  (Probability=0.828, Value=0.466)


Tuned Gradient Boosting: avg R2=0.650  (Probability=0.855, Value=0.445)
best: Tuned Gradient Boosting


In [ ]:
X = pd.get_dummies(df[input_features], columns=cat_cols, drop_first=True)
y = df[target_cols].copy()
y.loc[df["Purchased"] == "No", "Purchase_Value"] = 0

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
def evaluate_multi(model, name):
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    r2 = r2_score(y_test, pred, multioutput="uniform_average")
    per = r2_score(y_test, pred, multioutput="raw_values")
    print(f"{name}: avg R2={r2:.3f}  (Probability={per[0]:.3f}, Value={per[1]:.3f})")
    return model, r2


In [ ]:
lr_model, lr_r2 = evaluate_multi(LinearRegression(), "Linear Regression")

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), {"n_estimators":[100,200], "max_depth":[10,15]}, cv=2, scoring="r2")
rf_grid.fit(X_train_s, y_train)
rf_model, rf_r2 = evaluate_multi(rf_grid.best_estimator_, "Tuned Random Forest")

gb_grid = GridSearchCV(MultiOutputRegressor(GradientBoostingRegressor(random_state=42)),
                        {"estimator__n_estimators":[100], "estimator__learning_rate":[0.05,0.1], "estimator__max_depth":[2,3]},
                        cv=2, scoring="r2")
gb_grid.fit(X_train_s, y_train)
gb_model, gb_r2 = evaluate_multi(gb_grid.best_estimator_, "Tuned Gradient Boosting")

best_reg_name, best_reg_model = max(
    [("Linear Regression", lr_model), ("Tuned Random Forest", rf_model), ("Tuned Gradient Boosting", gb_model)],
    key=lambda t: r2_score(y_test, t[1].predict(X_test_s), multioutput="uniform_average"))
print("best:", best_reg_name)


In [3]:
import joblib
joblib.dump(best_reg_model, "../models/purchase_multi_output_model.pkl")
joblib.dump(scaler, "../models/purchase_reg_scaler.pkl")
joblib.dump(list(X.columns), "../models/purchase_reg_columns.pkl")
joblib.dump(target_cols, "../models/purchase_reg_targets.pkl")
print("saved:", best_reg_name)


saved: Tuned Gradient Boosting


## Part 2 - Classification (Purchase_Intent: High/Medium/Low)
Not using Purchase_Probability as a feature - Purchase_Intent is literally binned from
it (Low<0.35, Medium 0.35-0.65, High>=0.65), so that would be a 100% accuracy cheat.
Using class_weight="balanced" since Medium (368 rows) is way underrepresented vs
Low (1322 rows).

In [4]:
Xc = pd.get_dummies(df[input_features], columns=cat_cols, drop_first=True)
yc = df["Purchase_Intent"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
clf_scaler = MinMaxScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)
Xc_test_s = clf_scaler.transform(Xc_test)

def evaluate_clf(model, name):
    model.fit(Xc_train_s, yc_train)
    pred = model.predict(Xc_test_s)
    acc = accuracy_score(yc_test, pred)
    print(f"{name}: accuracy={acc:.3f}")
    return model, acc, pred

lr_clf, lr_acc, _ = evaluate_clf(LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
gb_clf, gb_acc, gb_pred = evaluate_clf(GradientBoostingClassifier(n_estimators=200, random_state=42), "Gradient Boosting")

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, class_weight="balanced"),
                        {"n_estimators":[100,200], "max_depth":[8,15]}, cv=2, scoring="f1_macro")
rf_grid.fit(Xc_train_s, yc_train)
rf_clf, rf_acc, rf_pred = evaluate_clf(rf_grid.best_estimator_, "Tuned Random Forest")

best_clf_name, best_clf_model, best_pred = max(
    [("Logistic Regression", lr_clf, None), ("Gradient Boosting", gb_clf, gb_pred), ("Tuned Random Forest", rf_clf, rf_pred)],
    key=lambda t: accuracy_score(yc_test, t[1].predict(Xc_test_s)))
print("best:", best_clf_name)
print(classification_report(yc_test, best_clf_model.predict(Xc_test_s)))


Logistic Regression: accuracy=0.781


Gradient Boosting: accuracy=0.840


Tuned Random Forest: accuracy=0.829
best: Gradient Boosting
              precision    recall  f1-score   support

        High       0.85      0.86      0.85       710
         Low       0.89      0.95      0.92      1322
      Medium       0.55      0.40      0.46       368

    accuracy                           0.84      2400
   macro avg       0.76      0.74      0.75      2400
weighted avg       0.83      0.84      0.83      2400



In [5]:
joblib.dump(best_clf_model, "../models/purchase_intent_classifier.pkl")
joblib.dump(clf_scaler, "../models/purchase_intent_clf_scaler.pkl")
joblib.dump(list(Xc.columns), "../models/purchase_intent_clf_columns.pkl")
print("saved:", best_clf_name)


saved: Gradient Boosting
